# 🧱 25 · Volumetric & Large-Scale Bioimage Analysis
### What changes when your image is a multi-gigabyte 3-D or 4-D volume

**Part J · Research frontier** · Module 25 of 29 (frontier) · builds on 11, 20

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/xulabs/edu/blob/main/sci_img/tutorial/BiologyToBioimage/notebooks/25_volumetric_and_large_scale_imaging.ipynb)

| | |
|---|---|
| **Hardware target** | CPU (small synthetic volumes only; real light-sheet/EM-scale data needs a GPU and often a cluster — discussed conceptually) |
| **Core frameworks** | NumPy, SciPy, PyTorch |
| **Prerequisites** | Module 11 (deep segmentation), Module 20 (foundation models) |
| **Builds on** | Modules 11, 20 |
| **Research frontier covered** | 3-D CNNs, patch-based training and stitching, chunked multiscale storage, distributed processing |
| **Estimated time** | 70–90 min |

> ⚠️ **Important notes.** For research and educational purposes only, not to be used for clinical diagnosis.
> **All data are SIMULATED.** The 3-D volume, every nucleus inside it and its exact instance mask are generated by code, so the Dice scores and the chunking-equivalence check below are exact, not estimated.

## 🎯 Learning objectives

By the end of this notebook you will be able to:

- Explain why a 2-D segmentation pipeline does not simply "become" 3-D for free — in memory, in
  compute, and in how much of the volume a model can see at once.
- Build and train a small 3-D CNN on randomly cropped sub-volumes, then reassemble whole-volume
  predictions by sliding-window inference with overlap-based stitching.
- Explain the idea behind chunked, multiscale, cloud-friendly storage (the OME-Zarr/NGFF idea),
  and prove with a toy experiment that chunk-wise processing can give the *exact* same answer as
  processing a whole array at once.
- Describe why distributed, cluster-scale processing becomes necessary once a volume no longer
  fits on one machine, and relate a toy parallel-chunk demo to how that works in practice.

## 🗺️ Contents

1. Why 3-D/4-D bioimages are a different scale problem
2. Building a synthetic 3-D volume with known ground truth
3. Patch-based training and sliding-window stitching
4. Chunked, multiscale storage: the OME-Zarr idea
5. Distributed, cluster-scale processing
6. 🧭 From toy to real data

## 🧫 The biological question

Every image so far in this course has been one 2-D plane. Real microscopy is rarely that
generous: a confocal z-stack, a light-sheet acquisition of a whole zebrafish embryo, or an
expansion-microscopy volume of a slab of brain tissue is a 3-D (or, with time, 4-D) array —
routinely tens to hundreds of gigabytes, sometimes terabytes, for a single acquisition. You cannot
just hand that to the 2-D U-Net from Module 11 and expect it to work: it cannot even load into
memory, let alone be processed one pixel-grid pass at a time. This module is about the three
things that change once "image" means "volume": the model architecture, the memory/compute
budget, and the storage and processing strategy.

---

## Environment setup

In [ ]:
# ── Hardware check ──
import platform
print('Python', platform.python_version())
try:
    import torch
    print('Torch', torch.__version__, '| CUDA available:', torch.cuda.is_available())
except ImportError:
    print('PyTorch not installed yet — the next cell installs it if needed.')


In [ ]:
!pip install -q scikit-image torch
# ── Dependencies ── (Colab already has most of these; we install only what is missing)


In [ ]:
# ── Global imports and configuration ──
import numpy as np
import matplotlib.pyplot as plt
from scipy import ndimage as ndi
SEED = 25
rng = np.random.default_rng(SEED)
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.titlesize'] = 11


### 🧰 Course toolkit

The exact same helper functions used throughout this course, copied verbatim so this notebook runs on its own.

In [ ]:
# ── Course toolkit (identical to ../course_toolkit.py v1.0; copied so this notebook runs alone) ──
import numpy as np
import matplotlib.pyplot as plt
from scipy import ndimage as ndi
from scipy.optimize import linear_sum_assignment

def add_gaussian_noise(image: np.ndarray, sigma: float, rng: np.random.Generator) -> np.ndarray:
    """Additive Gaussian (read) noise: the classic noise floor of a camera sensor."""
    return image + rng.normal(0.0, sigma, size=image.shape)


def show_image(ax, image: np.ndarray, title: str = "", cmap: str = "gray"):
    """Show one image on a given Matplotlib axis, titled, with ticks removed."""
    ax.imshow(image, cmap=cmap)
    ax.set_title(title, fontsize=10)
    ax.set_xticks([]); ax.set_yticks([])


def show_image_grid(images, titles=None, ncols: int = 4, cmap: str = "gray", figsize_scale: float = 2.6):
    """A grid of images with Matplotlib, one subplot each, titles optional."""
    n = len(images)
    ncols = min(ncols, n)
    nrows = int(np.ceil(n / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(figsize_scale * ncols, figsize_scale * nrows))
    axes = np.atleast_1d(axes).ravel()
    for i, img in enumerate(images):
        show_image(axes[i], img, titles[i] if titles else "", cmap=cmap)
    for ax in axes[n:]:
        ax.axis("off")
    fig.tight_layout()
    return fig, axes


def dice_coefficient(mask_a: np.ndarray, mask_b: np.ndarray) -> float:
    """Dice/F1 overlap of two binary masks: 2|A∩B| / (|A|+|B|)."""
    a, b = mask_a.astype(bool), mask_b.astype(bool)
    inter = (a & b).sum()
    denom = a.sum() + b.sum()
    return float(2 * inter / denom) if denom > 0 else 1.0

---

## 1 · Why 3-D/4-D bioimages are a different scale problem

A 2-D image's memory grows with its *area*; a 3-D volume's memory grows with its *volume* — the
cube, not the square, of linear resolution. Double the resolution along every axis and a 2-D
image needs 4× the memory; a 3-D volume needs 8×. A CNN makes this worse: every convolutional
layer keeps an *activation* map roughly the size of its input, and a 3-D convolution's activation
maps are exactly this same cube-growth, now multiplied across every channel and every layer of the
network.

In [ ]:
# ── Make the cube-growth concrete with real numbers ──
def volume_bytes(shape, dtype_bytes=4):
    return np.prod(shape) * dtype_bytes


def describe(shape, label):
    b = volume_bytes(shape)
    print(f"{label:28s} {str(shape):18s} {b/1e6:9.1f} MB")


print("Single float32 volume, no model yet:")
describe((160, 160), "one 2-D plane (160²)")
describe((160, 160, 160), "one 3-D cube (160³)")
describe((320, 320, 320), "double the resolution (320³)")
describe((1024, 1024, 500), "one modest confocal z-stack")
describe((2000, 2000, 2000), "one not-unusual light-sheet volume")

# A rough activation-memory budget: a 3-D U-Net-style encoder keeps several feature maps alive
# at once; a handful of 32-channel 3-D activation volumes at full resolution already dwarfs the
# raw image itself.
n_feature_maps, n_channels = 4, 32
raw = volume_bytes((320, 320, 320))
activations = volume_bytes((320, 320, 320)) * n_channels * n_feature_maps
print(f"\nraw 320³ volume            : {raw/1e6:9.1f} MB")
print(f"≈{n_feature_maps} feature maps × {n_channels} channels at full res: "
      f"{activations/1e6:9.1f} MB  ({activations/raw:.0f}× the raw data)")

🔍 **What just happened?** The numbers above are why nobody runs a 3-D CNN over an
entire multi-gigabyte volume in one forward pass, even on a large GPU: the *raw data* might just
about fit, but the *activations* needed to process it do not. The standard fix, which the rest of
this module builds end to end, is to train and run inference on small random **crops** (patches)
of the volume rather than the whole thing at once.

---

## 2 · Building a synthetic 3-D volume with known ground truth

We extend the 2-D nucleus-drawing idea from `simulate_nuclei_image` directly into 3-D: place
roughly spherical blobs (nuclei) at random 3-D positions, blur with an (anisotropic — z-resolution
is usually coarser than xy-resolution in real microscopes) point-spread function, and add noise.
This is a genuinely new helper (there is no 3-D version in the shared toolkit, since the rest of
the course stays 2-D), written directly in this notebook.

In [ ]:
# ── A small synthetic 3-D fluorescence volume, with exact instance-level ground truth ──
def simulate_nuclei_volume(n_cells=18, shape=(40, 56, 56), psf_sigma=(1.0, 1.3, 1.3),
                            background=0.05, noise_sigma=0.03, seed=0):
    """A SIMULATED 3-D volume (Z, Y, X) of roughly spherical nuclei, anisotropically blurred
    (z usually has coarser resolution than xy in real microscopes). Returns `volume`,
    `instance_mask` (0 = background, 1..n_cells) and `centroids` (z, y, x)."""
    rng = np.random.default_rng(seed)
    Z, Y, X = shape
    zz, yy, xx = np.meshgrid(np.arange(Z), np.arange(Y), np.arange(X), indexing="ij")
    volume = np.full(shape, background, dtype=float)
    instance_mask = np.zeros(shape, dtype=int)
    margin_z, margin_y, margin_x = Z * 0.18, Y * 0.14, X * 0.14
    min_sep = min(Y, X) * 0.17
    placed, attempts = [], 0
    while len(placed) < n_cells and attempts < n_cells * 100:
        attempts += 1
        cz = rng.uniform(margin_z, Z - margin_z)
        cy = rng.uniform(margin_y, Y - margin_y)
        cx = rng.uniform(margin_x, X - margin_x)
        if all(np.hypot(cy - py, cx - px) > min_sep for _, py, px in placed):
            placed.append((cz, cy, cx))
    centroids = []
    for i, (cz, cy, cx) in enumerate(placed):
        r = rng.uniform(min(Y, X) * 0.055, min(Y, X) * 0.08)
        d = ((zz - cz) / (r * 0.75)) ** 2 + ((yy - cy) / r) ** 2 + ((xx - cx) / r) ** 2
        blob = np.exp(-d * 2.0)
        volume += blob
        mask = blob > 0.25
        instance_mask[mask & (instance_mask == 0)] = i + 1
        centroids.append((cz, cy, cx))
    volume = ndi.gaussian_filter(volume, sigma=psf_sigma)
    volume = add_gaussian_noise(volume, noise_sigma, rng)
    volume = np.clip(volume, 0, None)
    return dict(volume=volume, instance_mask=instance_mask,
                centroids=np.array(centroids), source="SIMULATED")


VOL_SHAPE = (40, 56, 56)
sim3d = simulate_nuclei_volume(n_cells=16, shape=VOL_SHAPE, seed=SEED)
volume, instance_mask = sim3d["volume"], sim3d["instance_mask"]
print("volume shape:", volume.shape, "| size:", f"{volume_bytes(volume.shape)/1e6:.2f} MB",
      "| cells:", instance_mask.max())

fig, axes = plt.subplots(1, 4, figsize=(12, 3))
for ax, z in zip(axes, np.linspace(5, VOL_SHAPE[0] - 6, 4).astype(int)):
    show_image(ax, volume[z], f"z = {z}")
fig.suptitle("Four z-slices through the simulated volume")
fig.tight_layout()
plt.show()

🔍 **What just happened?** Walking through z-slices, you can see individual nuclei
come into and out of focus — exactly like scrolling through a real confocal z-stack in Fiji's
image viewer. Because we generated this volume ourselves, `instance_mask` is an exact, pixel-
perfect 3-D ground truth we can check any segmentation against below.

---

## 3 · Patch-based training and sliding-window stitching

We will never show the whole volume to the network at once (§1 explained why). Instead:

1. **Train** on small random 3-D crops ("patches"), each labeled foreground/background from the
   matching crop of `instance_mask`.
2. **Infer** by sliding a window across the *whole* volume, patch by patch, and stitching the
   per-patch predictions back into a full-volume prediction.

In [ ]:
# ── A tiny 3-D CNN, trained on random crops ──
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(SEED)
PATCH = (16, 24, 24)


class Tiny3DSegNet(nn.Module):
    """A minimal 3-D CNN: predicts a per-voxel foreground/background logit for a (Z,Y,X) crop."""

    def __init__(self, c=8):
        super().__init__()
        self.conv1 = nn.Conv3d(1, c, 3, padding=1)
        self.conv2 = nn.Conv3d(c, c, 3, padding=1)
        self.conv3 = nn.Conv3d(c, 1, 1)

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        return self.conv3(x)          # raw logits, same (Z,Y,X) shape as the input


def random_crop(volume, mask, patch_shape, rng):
    Z, Y, X = volume.shape
    pz, py, px = patch_shape
    z0 = rng.integers(0, Z - pz + 1); y0 = rng.integers(0, Y - py + 1); x0 = rng.integers(0, X - px + 1)
    sl = (slice(z0, z0 + pz), slice(y0, y0 + py), slice(x0, x0 + px))
    return volume[sl], mask[sl]


def dice_loss(logits, target):
    prob = torch.sigmoid(logits)
    inter = (prob * target).sum()
    return 1 - (2 * inter + 1) / (prob.sum() + target.sum() + 1)


def train_3d_segmenter(volume, instance_mask, n_steps=200, lr=1e-3, seed=0):
    torch.manual_seed(seed)
    rng_local = np.random.default_rng(seed)
    model = Tiny3DSegNet()
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    fg = (instance_mask > 0).astype(np.float32)
    for step in range(n_steps):
        crop_vol, crop_fg = random_crop(volume, fg, PATCH, rng_local)
        x = torch.from_numpy(crop_vol[None, None]).float()
        y = torch.from_numpy(crop_fg[None, None]).float()
        opt.zero_grad()
        loss = F.binary_cross_entropy_with_logits(model(x), y) + dice_loss(model(x), y)
        loss.backward()
        opt.step()
    return model


seg_model = train_3d_segmenter(volume, instance_mask, seed=SEED)
print("trained a tiny 3-D segmenter on 200 random 16x24x24 crops")

In [ ]:
# ── Sliding-window inference with overlap-add stitching over the WHOLE volume ──
def sliding_window_infer_3d(model, volume, patch_shape, stride=None):
    """Run `model` on overlapping patches across the whole volume and stitch the results by
    averaging overlapping predictions (overlap-add) — the standard way to apply a patch-trained
    3-D network to a volume too large to process in one pass."""
    Z, Y, X = volume.shape
    pz, py, px = patch_shape
    stride = stride or (pz // 2, py // 2, px // 2)
    sz, sy, sx = stride
    accum = np.zeros(volume.shape, dtype=float)
    counts = np.zeros(volume.shape, dtype=float)
    z_starts = list(range(0, Z - pz + 1, sz)) + [Z - pz]
    y_starts = list(range(0, Y - py + 1, sy)) + [Y - py]
    x_starts = list(range(0, X - px + 1, sx)) + [X - px]
    model.eval()
    with torch.no_grad():
        for z0 in sorted(set(z_starts)):
            for y0 in sorted(set(y_starts)):
                for x0 in sorted(set(x_starts)):
                    sl = (slice(z0, z0 + pz), slice(y0, y0 + py), slice(x0, x0 + px))
                    crop = volume[sl]
                    logits = model(torch.from_numpy(crop[None, None]).float())[0, 0].numpy()
                    accum[sl] += logits
                    counts[sl] += 1.0
    return accum / np.maximum(counts, 1e-9)


full_logits = sliding_window_infer_3d(seg_model, volume, PATCH)
full_pred = full_logits > 0
true_fg = instance_mask > 0
dice = dice_coefficient(full_pred, true_fg)
print(f"whole-volume Dice (patch-trained, stitched prediction vs. exact ground truth): {dice:.3f}")

fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for col, z in enumerate(np.linspace(5, VOL_SHAPE[0] - 6, 4).astype(int)):
    show_image(axes[0, col], true_fg[z].astype(float), f"ground truth, z={z}")
    show_image(axes[1, col], full_pred[z].astype(float), f"stitched prediction, z={z}")
fig.tight_layout()
plt.show()

🧪 **Ground-truth check.** The Dice score above compares the *stitched, whole-volume*
prediction — built entirely from small patches the network has never seen assembled together —
against the exact instance mask we created the volume from. A score well above chance (which
would be near 0 for a sparse foreground) confirms the patch-and-stitch strategy recovers a
volume-level answer from only ever looking at small pieces at a time.

✏️ **Your turn.** Re-run `sliding_window_infer_3d` with `stride=PATCH` (i.e. no
overlap at all) instead of the default half-patch overlap. Compare the Dice score and look at the
slice images near patch boundaries — non-overlapping tiling tends to show visible seams because
each patch's edge pixels got less context than its center; overlap-add softens this by blending
multiple overlapping predictions together, which is exactly why real volumetric pipelines
(Module 20's foundation models included) always use overlapping patches with a blending halo.

---

## 4 · Chunked, multiscale storage: the OME-Zarr idea

Patch-based *processing* solves the compute problem. There is a matching *storage* problem: you
do not want to have to read an entire multi-gigabyte volume into memory just to look at one
corner of it, or just to show a quick low-resolution overview. The OME-Zarr / NGFF ("Next
Generation File Format") approach solves both at once:

- The array is stored as independent **chunks** (e.g. 64×64×64 blocks) that can each be read or
  written without touching the rest of the array.
- Several **resolution levels** (a pyramid: full-res, then 2×, 4×, 8× downsampled, …) are stored
  side by side, so a viewer can show a fast overview at low resolution and only fetch full-
  resolution chunks for the region you actually zoom into.

The key correctness property — the one that makes chunking trustworthy rather than just
convenient — is that for anything expressible as a sum over voxels (a mean, a total intensity, a
histogram, …), processing chunk by chunk and combining the partial results gives **exactly** the
same answer as processing the whole array at once. Let's prove that, rather than just assert it.

In [ ]:
# ── 🧪 Prove chunked processing gives the exact same answer as processing the whole volume ──
def iter_chunks(shape, chunk_shape):
    """Yield non-overlapping chunk slices tiling `shape` with blocks of `chunk_shape`."""
    ranges = [range(0, s, c) for s, c in zip(shape, chunk_shape)]
    for z0 in ranges[0]:
        for y0 in ranges[1]:
            for x0 in ranges[2]:
                yield tuple(slice(o, min(o + c, s)) for o, c, s in
                            zip((z0, y0, x0), chunk_shape, shape))


def chunked_mean(volume, chunk_shape):
    total, count = 0.0, 0
    for sl in iter_chunks(volume.shape, chunk_shape):
        block = volume[sl]
        total += block.sum()
        count += block.size
    return total / count


whole_mean = volume.mean()
chunk_mean = chunked_mean(volume, (16, 16, 16))
print(f"mean computed on the whole volume at once : {whole_mean:.8f}")
print(f"mean computed chunk-by-chunk (16³ blocks) : {chunk_mean:.8f}")
print(f"difference                                : {abs(whole_mean - chunk_mean):.2e}  "
      f"(floating-point round-off only)")

# A multiscale pyramid: a 2x-downsampled "overview" level, built without ever holding two full
# copies of the full-resolution array side by side in the final product.
overview = volume[::2, ::2, ::2]
print(f"\nfull-resolution volume : {volume.shape}, {volume_bytes(volume.shape)/1e6:.2f} MB")
print(f"overview (level 1)     : {overview.shape}, {volume_bytes(overview.shape)/1e6:.2f} MB "
      f"({volume_bytes(volume.shape) / volume_bytes(overview.shape):.0f}x smaller)")

🔍 **What just happened?** The two means agree to floating-point precision — chunking
a *reduction* (sum, mean, histogram) is exact, not an approximation, as long as you combine the
partial results correctly. This is why tools built on chunked arrays (Zarr, Dask) can compute
statistics over a volume far larger than memory without ever materializing the whole thing at
once. The overview level is simply a strided (or, in a real pipeline, box-averaged) subsample —
small enough to page in instantly for a quick look, while the full-resolution chunks stay on disk
until you actually zoom in.

---

## 5 · Distributed, cluster-scale processing

Once a volume is too large even for one machine's RAM — routine for whole-brain light-sheet or
large EM datasets — the same chunk-based thinking extends naturally to *multiple machines*: each
worker in a cluster is handed a subset of chunks, processes them independently (exactly like
`iter_chunks` above, just farmed out instead of looped over), and the partial results are combined
at the end. The toy demo below uses Python's own `concurrent.futures` to run several chunks "in
parallel" on one machine — a stand-in for what a real cluster scheduler (Dask distributed, Spark,
or a custom pipeline) does across many machines.

In [ ]:
# ── A toy stand-in for distributed chunk processing ──
import concurrent.futures as cf
import time


def _chunk_sum(args):
    volume, sl = args
    return float(volume[sl].sum()), volume[sl].size


chunks = list(iter_chunks(volume.shape, (16, 16, 16)))
t0 = time.time()
with cf.ThreadPoolExecutor(max_workers=4) as pool:
    results = list(pool.map(_chunk_sum, [(volume, sl) for sl in chunks]))
t_parallel = time.time() - t0

total = sum(r[0] for r in results)
count = sum(r[1] for r in results)
print(f"{len(chunks)} chunks processed by a 4-worker pool in {t_parallel*1000:.1f} ms")
print(f"recombined mean: {total / count:.8f}  (matches the whole-volume mean above: {whole_mean:.8f})")

🔍 **What just happened?** Four workers computed one chunk's partial sum each (in
whatever order the scheduler assigned), and recombining those partial sums reproduces the exact
whole-volume mean — the same correctness guarantee as §4, now spread across workers instead of a
single loop. A real distributed array library (Dask, for instance) builds a *task graph* out of
exactly this chunk-sum-then-combine pattern automatically, and can send the chunk tasks to
different physical machines rather than threads on one laptop; the correctness argument does not
change, only who executes each task.

---

## 6 · 🧭 From toy to real data

### 📄 Research connection — 3-D U-Net
Çiçek, Ö., Abdulkadir, A., Lienkamp, S. S., Brox, T. & Ronneberger, O. (2016). *3D U-Net: Learning
Dense Volumetric Segmentation from Sparse Annotation.* MICCAI. The direct 3-D generalization of
Module 11's U-Net, trained with exactly the patch-based strategy of §3 because full 3-D volumes
essentially never fit in GPU memory at once.

### 📄 Research connection — OME-Zarr / NGFF
Moore, J. et al. (2021). *OME-NGFF: a next-generation file format for expanding bioimaging data-
access strategies and use cases.* Histochemistry and Cell Biology, 155. The real specification
behind §4's chunked, multiscale-pyramid idea — designed specifically so cloud storage and
browser-based viewers can serve a petabyte-scale dataset one small chunk at a time.

### 📄 Research connection — large-scale light-sheet pipelines
Royer, L. A. et al. (2016). *Adaptive light-sheet microscopy for long-term, high-resolution
imaging in living organisms.* Nature Biotechnology, 34. A concrete example of acquisitions large
enough that §1's memory argument is not academic — these pipelines are built around chunked,
patch-based processing by necessity, not by choice.

### 📄 Research connection — the BigDataViewer / Fiji ecosystem
Pietzsch, T. et al. (2015). *BigDataViewer: visualization and processing for large image data
sets.* Nature Methods, 12. The viewer-side counterpart to §4: lets a user scroll through a
dataset far larger than RAM by streaming exactly the chunks the current view needs, at the
resolution level the current zoom calls for.

At production scale, the toy volume in this notebook (40×56×56, under 1 MB) stands in for
acquisitions that are routinely gigabytes to terabytes; the code above (patch training, sliding-
window stitching, chunked reductions, multiscale pyramids) is the same code shape used on those
real volumes — only the chunk sizes, the cluster size, and the storage backend (Zarr on an object
store instead of a NumPy array in RAM) change.

## 📌 Key takeaways

- 3-D volumes grow memory with the *cube* of resolution, and a CNN's activations multiply that
  growth across channels and layers — full-volume processing runs out of memory almost
  immediately at real acquisition sizes.
- The standard fix is patch-based training plus sliding-window inference with overlap-add
  stitching; overlap avoids visible seams at patch boundaries.
- Chunked storage (the OME-Zarr/NGFF idea) lets you read, write and reduce over an array far
  larger than memory, one chunk at a time — and for sums/means/histograms this is provably exact,
  not approximate.
- Multiscale pyramids trade a small amount of extra storage for instant low-resolution overviews,
  without ever requiring the full-resolution array to be loaded just to look at it.
- Distributed processing is the same chunk-and-combine idea spread across machines instead of one
  loop — the correctness argument is identical.

## ✅ Checkpoint

**1. Why does a non-overlapping tiling of a volume tend to show visible seams in a patch-trained
segmentation network's output, while an overlapping tiling with averaging does not?**

<details><summary>Answer</summary>

A convolutional network's prediction near a patch's edge has less spatial context than a
prediction near its center (it never "sees" what is just outside the patch). With non-overlapping
tiles, every patch boundary is therefore a seam where context genuinely drops off, and it shows up
as a visible discontinuity. Overlapping patches with averaging mean every voxel near a would-be
seam gets contributions from at least one patch where it sits nearer the center (with full
context), and averaging smooths over the discrepancy.
</details>

**2. Is computing a chunk-by-chunk *mean* (as in §4) always exactly equal to the whole-array
mean, or could floating-point order-of-operations ever make it slightly different — and does that
matter?**

<details><summary>Answer</summary>

In exact (infinite-precision) arithmetic it is always exactly equal, because a mean over the
whole array is just a sum over all elements divided by the count, and summing is associative in
exact arithmetic regardless of grouping. In actual floating-point arithmetic, summing in a
different order can change the result by the same tiny rounding error you would get from adding
the same numbers in a different order on a calculator — far smaller than any measurement or
simulation uncertainty in this course, so it does not matter in practice.
</details>

**3. A lab has a 2 TB light-sheet dataset and a workstation with 64 GB of RAM. Name one thing
from this module they could do without ever loading the whole dataset into memory.**

<details><summary>Answer</summary>

Any of: compute a global statistic (mean intensity, a histogram) by processing chunk by chunk and
combining partial results (§4); segment the volume with a patch-trained 3-D network and sliding-
window stitching, one patch's worth of memory at a time (§3); or generate/view a low-resolution
multiscale overview without ever reading full-resolution chunks outside the region currently being
viewed (§4).
</details>

## 🧠 Questions & discussion

- A 3-D CNN trained on 16×24×24 patches has never once seen the relationship between a cell at
  one end of the volume and a cell at the other end. What kinds of biological questions could this
  architecture *never* answer correctly, no matter how it is trained, simply because of its
  limited field of view?
- Multiscale pyramids trade extra storage (roughly 33% more, for a standard octave pyramid) for
  faster overviews. When is that trade worth it, and when would you skip building the pyramid
  entirely?
- If your lab's cluster has 20 workers but your dataset only has 8 chunks, what happens to the
  other 12 workers — and how would you re-chunk the data to use the cluster more fully?

## 🚀 Bonus — going further

- Modify `sliding_window_infer_3d` to crop out only the *center* region of each patch's
  prediction (discarding a border halo) instead of overlap-averaging the whole patch, and compare
  the resulting Dice score and visual seams to the overlap-add version.
- Build a second, finer pyramid level (4× downsampled) and time how much faster a crude
  "find roughly where the cells are" pass runs on the overview level compared to full resolution.
- Replace the `ThreadPoolExecutor` in §5 with a `ProcessPoolExecutor` and compare timings — can
  you explain the difference given Python's Global Interpreter Lock?

## 📝 Lab note

Record, in your own words:

1. One number: the memory, in MB, of the `(2000, 2000, 2000)` volume computed in §1, and whether
   that would fit on your own laptop.
2. One sentence: why chunked reductions (mean, sum) are *exact* while patch-based deep-learning
   inference needs overlap to avoid seams — what is different about the two kinds of operation?
3. One claim backed by a number: the whole-volume Dice score your trained segmenter achieved in
   §3, and whether you consider that good enough for a real measurement.

## 📚 References

1. Çiçek, Ö. et al. (2016). 3D U-Net: Learning Dense Volumetric Segmentation from Sparse
   Annotation. *MICCAI*.
2. Moore, J. et al. (2021). OME-NGFF: a next-generation file format for expanding bioimaging
   data-access strategies and use cases. *Histochemistry and Cell Biology*, 155.
3. Royer, L. A. et al. (2016). Adaptive light-sheet microscopy for long-term, high-resolution
   imaging in living organisms. *Nature Biotechnology*, 34.
4. Pietzsch, T. et al. (2015). BigDataViewer: visualization and processing for large image data
   sets. *Nature Methods*, 12.
5. Moore, J. et al. (2023). OME-Zarr: a cloud-optimized bioimaging file format with international
   community support. *Histochemistry and Cell Biology*, 160 (the current community spec).
6. Rueden, C. T. et al. (2022). ImageJ2: ImageJ for the next generation of scientific image data.
   *BMC Bioinformatics* (the Fiji ecosystem referenced in §6).
7. Rocklin, M. (2015). Dask: Parallel Computation with Blocked algorithms and Task Scheduling.
   *SciPy Conference* (the real chunked-array/distributed-scheduler library behind §5's toy demo).

## ➡️ Next

Module 26 turns the uncertainty lens of Module 24 toward a sharper question: when a model is
wrong, *why* was it wrong, and how do you audit a deployed model for the kinds of silent failure
that no accuracy number on your own test set would ever reveal?